# Why Are Deliveries Late? — Part 2: SQL Analysis

Business questions answered in SQL against the cleaned data from Part 1 (`data/supply_chain.db`, table `order_items`).

**Key columns**
| Column | Meaning |
|---|---|
| `order_id` | one customer order (an order can have several items/rows) |
| `order_item_id` | one product line within an order — one row |
| `shipping_mode` | First Class, Second Class, Standard Class, Same Day |
| `days_shipping_scheduled` / `days_shipping_actual` | promised vs actual shipping days |
| `days_late` | actual − promised (negative = early) |
| `is_late` | 1 = late, 0 = not late, NULL = cancelled |
| `is_cancelled` | 1 = shipment cancelled |
| `market`, `order_region`, `order_country` | where the order was delivered |
| `customer_segment` | Consumer, Corporate, Home Office |
| `item_sales_net`, `item_profit` | revenue and profit for the item (after discount) |
| `order_date`, `order_month` | dates stored as text `YYYY-MM-DD HH:MM:SS` |

## Setup — connect to the database

In [4]:
%load_ext sql
%sql sqlite:///data/supply_chain.db
%config SqlMagic.displaylimit = 50

Connecting to 'sqlite:///data/supply_chain.db'

Every SQL cell starts with `%%sql` on the first line — after that, it's normal SQL.

**MySQL → SQLite differences to remember**
| MySQL | SQLite |
|---|---|
| `YEAR(order_date)` | `strftime('%Y', order_date)` |
| `DATE_FORMAT(order_date, '%Y-%m')` | `strftime('%Y-%m', order_date)` |
| `IF(cond, a, b)` | `CASE WHEN cond THEN a ELSE b END` (or `IIF(cond, a, b)`) |
| `CONCAT(a, b)` | `a \|\| b` |
| `SHOW COLUMNS FROM t` | `PRAGMA table_info(t)` |

Everything else — `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT`, `JOIN`, subqueries, `WITH` (CTEs), window functions — works the same.

### Warm-up: look at the table

In [17]:
%%sql
SELECT order_id, order_item_id, product_name, shipping_mode,
       days_shipping_scheduled, days_shipping_actual, days_late, is_late
FROM order_items
LIMIT 5;


Running query in 'sqlite:///data/supply_chain.db'

order_id,order_item_id,product_name,shipping_mode,days_shipping_scheduled,days_shipping_actual,days_late,is_late
77202,180517,Smart watch,Standard Class,4,3,-1,0
75939,179254,Smart watch,Standard Class,4,5,1,1
75938,179253,Smart watch,Standard Class,4,4,0,0
75937,179252,Smart watch,Standard Class,4,3,-1,0
75936,179251,Smart watch,Standard Class,4,2,-2,0


## Q1. How big is the data?
How many **rows (items)**, how many **distinct orders**, and how many **cancelled orders**?

In [18]:
%%sql
SELECT
    COUNT(*)                   AS total_items,
    COUNT(DISTINCT order_id)   AS total_orders,
    COUNT(DISTINCT CASE WHEN is_cancelled = 1 THEN order_id END) AS cancelled_orders,
    COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END) AS late_orders 
FROM order_items;

Running query in 'sqlite:///data/supply_chain.db'

total_items,total_orders,cancelled_orders,late_orders
180519,65752,2855,36048


**Finding:** The data has 180,519 order items across 65,752 orders; 2,855 orders were cancelled and 36,048 were delivered late. Because one order can contain several items, late rates are calculated on distinct orders, not rows.

## Q2. What is the overall late rate?
What % of **shipped orders** (not items, not cancelled) arrived late? Expected answer: **57.3%**.

In [24]:
%%sql
SELECT
    COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END) AS late_orders,
    COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END) AS shipped_orders,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
          / COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END), 1) AS late_pct
FROM order_items;


Running query in 'sqlite:///data/supply_chain.db'

late_orders,shipped_orders,late_pct
36048,62897,57.3


**Finding:** 57.3% of shipped orders (36,048 of 62,897) were delivered late.

## Q3. Which shipping mode is late most often?
For each `shipping_mode`: late orders, shipped orders and late rate %. Sort by late rate.

In [31]:
%%sql
SELECT 
    shipping_mode,
    COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END) AS late_orders,
    COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END) AS shipped_orders,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
          / COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END), 1) AS late_pct
FROM order_items
group by shipping_mode
ORDER BY late_pct DESC;

Running query in 'sqlite:///data/supply_chain.db'

shipping_mode,late_orders,shipped_orders,late_pct
First Class,9602,9602,100.0
Second Class,9803,12256,80.0
Same Day,1648,3407,48.4
Standard Class,14995,37632,39.8


**Finding:** First Class is late on 100% of orders and Second Class on 80%, while Standard Class is late on only 40%. Premium shipping is the problem.

## Q4. Is it a geography problem?
Late rate % by `market`. Are some markets much worse than others?

In [32]:
%%sql
SELECT 
    market,
    COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END) AS late_orders,
    COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END) AS shipped_orders,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
          / COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END), 1) AS late_pct
FROM order_items
group by market
ORDER BY late_pct DESC;

Running query in 'sqlite:///data/supply_chain.db'

market,late_orders,shipped_orders,late_pct
Pacific Asia,9720,16853,57.7
USCA,4704,8178,57.5
Europe,10199,17763,57.4
LATAM,9339,16414,56.9
Africa,2086,3689,56.5


**Finding:** Late rate is about 57% in every market (56.5%–57.7%), so geography is not the cause.

## Q5. How late is late?
For **late orders only**, the average `days_late` by shipping mode.

In [36]:
%%sql
SELECT
    shipping_mode,
    ROUND(AVG(days_late), 1) AS avg_days_late
FROM order_items
WHERE is_late = 1
GROUP BY shipping_mode
ORDER BY avg_days_late DESC;

Running query in 'sqlite:///data/supply_chain.db'

shipping_mode,avg_days_late
Second Class,2.5
Standard Class,1.5
Same Day,1.0
First Class,1.0


**Finding:** When Second Class is late, it is ~2 days late on average, the worst of all modes. First Class is always exactly 1 day late.

## Q6. What is the business impact?
Total revenue (`item_sales_net`) from late orders vs all shipped orders, by shipping mode. How much premium-shipping revenue arrives late?

In [43]:
%%sql
SELECT
    shipping_mode,
    ROUND(SUM(item_sales_net), 0) AS total_revenue,
    ROUND(SUM(CASE WHEN is_late = 1 THEN item_sales_net END), 0) AS late_revenue,
    ROUND(100.0 * SUM(CASE WHEN is_late = 1 THEN item_sales_net END)
          / SUM(item_sales_net), 1) AS late_revenue_pct
FROM order_items
WHERE is_cancelled = 0
GROUP BY shipping_mode
ORDER BY late_revenue DESC;

Running query in 'sqlite:///data/supply_chain.db'

shipping_mode,total_revenue,late_revenue,late_revenue_pct
Standard Class,18956374.0,7516306.0,39.7
Second Class,6160631.0,4918063.0,79.8
First Class,4860950.0,4860950.0,100.0
Same Day,1666711.0,787237.0,47.2


**Finding:** $18.1M of $31.6M revenue (57%) came from late orders. Standard Class has the most late revenue ($7.5M) due to volume, while 100% of First Class revenue ($4.9M) arrived late.

## Q7. Is it getting better or worse over time?
Late rate % by month (`order_month`). Remember: data after Sept 2017 is incomplete.

In [49]:
%%sql
SELECT order_month,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT CASE WHEN is_cancelled=0 THEN order_id END), 1) AS late_pct
FROM order_items
WHERE is_cancelled = 0
GROUP BY order_month
ORDER BY order_month asc;

Running query in 'sqlite:///data/supply_chain.db'

order_month,late_pct
2015-01-01 00:00:00,56.2
2015-02-01 00:00:00,57.9
2015-03-01 00:00:00,57.5
2015-04-01 00:00:00,56.6
2015-05-01 00:00:00,57.5
2015-06-01 00:00:00,57.0
2015-07-01 00:00:00,56.5
2015-08-01 00:00:00,57.6
2015-09-01 00:00:00,58.2
2015-10-01 00:00:00,57.4


**Finding:** The late rate stayed between 55% and 60% every month for three years. This is a persistent, structural problem, not a seasonal one, and it has not improved.

## Q8. Which countries suffer most?
Top 10 destination countries by late rate, **only countries with at least 500 shipped orders**.

In [56]:
%%sql
SELECT order_country,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT CASE WHEN is_cancelled=0 THEN order_id END), 1) AS late_pct
FROM order_items
GROUP BY order_country
HAVING COUNT(DISTINCT CASE WHEN is_cancelled = 0 THEN order_id END) >= 500
ORDER BY late_pct desc
limit 10;

Running query in 'sqlite:///data/supply_chain.db'

order_country,late_pct
Nicaragua,59.6
Turkey,58.4
Germany,58.4
France,58.4
India,58.3
Indonesia,58.1
El Salvador,57.9
Philippines,57.8
Mexico,57.8
United States,57.7


**Finding:** Even the worst countries (Nicaragua 59.6%) are only ~2 points above the 57.3% average. Late delivery is not a country-level problem; it is driven by shipping mode.